# Modulo 7: Aplicacoes de Mineracao de Dados
## Social CRM, Web Mining e Analise de Sentimentos


---

### Objetivos de Aprendizagem
Ao final deste notebook voce sera capaz de:
- Analisar sentimentos com VADER (abordagem lexica) e Machine Learning
- Construir um pipeline completo de NLP em portugues
- Implementar sistemas de recomendacao com Filtragem Colaborativa (Surprise) e por Conteudo
- Segmentar clientes com o modelo RFM + K-Means (Social CRM)
- Prever churn com XGBoost e interpretar com feature importance
- Implementar PageRank e analisar redes com NetworkX
- Calcular centralidades e detectar comunidades em redes sociais

---

### Sumario
1. Analise de Sentimentos - VADER (Lexicon-based)
2. Analise de Sentimentos com Machine Learning
3. NLP Pipeline - Pre-processamento de Texto
4. Sistema de Recomendacao - Filtragem Colaborativa
5. Sistema de Recomendacao - Filtragem por Conteudo
6. Social CRM - Segmentacao de Clientes (RFM)
7. Web Mining - Analise de Texto e PageRank
8. Mineracao de Redes Sociais
9. Exercicios

In [ ]:
# Instalacao das dependencias
!pip install -q transformers torch scikit-learn pandas numpy matplotlib seaborn scipy vaderSentiment scikit-surprise requests beautifulsoup4 nltk wordcloud networkx plotly

## Setup - Imports e Configuracoes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import random

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Estilo global
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='deep')

# Cores do modulo
MINEBLUE = '#005AB4'
MINEORANGE = '#E67300'
MINEGREEN = '#228B22'
MINERED = '#B4142A'

# Downloads NLTK
import nltk
for corpus in ['stopwords', 'punkt', 'wordnet', 'rslp', 'averaged_perceptron_tagger']:
    nltk.download(corpus, quiet=True)

print('Setup concluido com sucesso!')

## 1. Analise de Sentimentos - VADER (Lexicon-based)

VADER (Valence Aware Dictionary and sEntiment Reasoner) e uma ferramenta de analise de sentimentos baseada em regras e lexico, especialmente eficaz para textos de redes sociais. Ele nao requer treinamento e lida com pontuacao, emojis e girias.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# 20 sentencas de exemplo (mistura de portugues e ingles)
frases = [
    # Positivas em ingles (VADER e otimizado para ingles)
    ("This product is absolutely amazing! Best purchase ever!!!", 'Positivo'),
    ("I love this movie, it was fantastic and heartwarming.", 'Positivo'),
    ("Outstanding service, highly recommend to everyone!", 'Positivo'),
    ("The food was delicious and the staff were very friendly.", 'Positivo'),
    ("GREAT experience! Would definitely come back :)", 'Positivo'),
    # Negativas em ingles
    ("This is the worst product I have ever bought. Terrible quality.", 'Negativo'),
    ("Absolutely awful! Complete waste of money. DO NOT BUY!", 'Negativo'),
    ("The service was horrible and the staff were rude.", 'Negativo'),
    ("I hate this app, it crashes every single time.", 'Negativo'),
    ("Disgusting food, never going back there again.", 'Negativo'),
    # Neutras
    ("The product arrived yesterday and I opened the box.", 'Neutro'),
    ("The meeting is scheduled for Tuesday at 3pm.", 'Neutro'),
    ("The package weighs approximately 2 kilograms.", 'Neutro'),
    # Ironia / dificuldades
    ("Oh great, another Monday morning...", 'Negativo'),  # ironico
    ("Yeah right, like that's going to work out well.", 'Negativo'),  # sarcasmo
    # Portugues (VADER tem limitacoes em PT)
    ("Este produto e absolutamente incrivel! Adorei!", 'Positivo'),
    ("Pessimo servico, nunca mais volto a este lugar.", 'Negativo'),
    ("O pacote chegou ontem sem danos.", 'Neutro'),
    ("Nao gostei nem um pouco do atendimento.", 'Negativo'),
    ("Muito bom! Recomendo para todos os meus amigos!", 'Positivo')
]

# Analisar com VADER
resultados = []
for frase, rotulo_real in frases:
    scores = analyzer.polarity_scores(frase)
    compound = scores['compound']
    if compound >= 0.05:
        pred = 'Positivo'
    elif compound <= -0.05:
        pred = 'Negativo'
    else:
        pred = 'Neutro'
    resultados.append({
        'Frase': frase[:55] + '...' if len(frase) > 55 else frase,
        'Rotulo Real': rotulo_real,
        'VADER Pred': pred,
        'Compound': round(compound, 3),
        'Positivo': round(scores['pos'], 3),
        'Negativo': round(scores['neg'], 3),
        'Neutro_score': round(scores['neu'], 3),
        'Correto': rotulo_real == pred
    })

df_vader = pd.DataFrame(resultados)
acuracia = df_vader['Correto'].mean()
print(f'Acuracia VADER nas 20 sentencas: {acuracia:.1%}')
print()
print(df_vader[['Frase','Rotulo Real','VADER Pred','Compound','Correto']].to_string(index=False))

# Grafico de barras dos scores compound
fig, ax = plt.subplots(figsize=(12, 6))
cores = [MINEGREEN if c >= 0.05 else (MINERED if c <= -0.05 else MINEORANGE)
         for c in df_vader['Compound']]
bars = ax.barh(range(len(df_vader)), df_vader['Compound'], color=cores, alpha=0.8)
ax.set_yticks(range(len(df_vader)))
ax.set_yticklabels([f'{r["Frase"][:40]}...' if len(r['Frase']) > 40 else r['Frase']
                    for _, r in df_vader.iterrows()], fontsize=7)
ax.axvline(x=0.05, color=MINEGREEN, linestyle='--', alpha=0.5, label='Limiar Positivo')
ax.axvline(x=-0.05, color=MINERED, linestyle='--', alpha=0.5, label='Limiar Negativo')
ax.set_xlabel('Score Compound VADER')
ax.set_title('Analise de Sentimentos com VADER - Scores Compound')
ax.legend()
plt.tight_layout()
plt.savefig('vader_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Distribuicao dos sentimentos
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribuicao
df_vader['VADER Pred'].value_counts().plot(
    kind='bar', ax=axes[0], color=[MINEGREEN, MINERED, MINEORANGE], alpha=0.8
)
axes[0].set_title('Distribuicao de Predicoes VADER')
axes[0].set_xlabel('Sentimento')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=0)

# Matriz de confusao
labels = ['Positivo', 'Negativo', 'Neutro']
cm = confusion_matrix(df_vader['Rotulo Real'], df_vader['VADER Pred'], labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Matriz de Confusao - VADER')

plt.tight_layout()
plt.show()

# Casos de falha
print('\n=== CASOS ONDE VADER FALHOU ===')
falhas = df_vader[~df_vader['Correto']]
for _, row in falhas.iterrows():
    print(f'  Real: {row["Rotulo Real"]:8s} | Pred: {row["VADER Pred"]:8s} | '
          f'Compound: {row["Compound"]:+.3f} | {row["Frase"]}')

print('\n=== ANALISE DOS PROBLEMAS ===')
print('1. Ironia/Sarcasmo: VADER nao detecta sentido oposto ao literal')
print('2. Portugues: VADER foi desenvolvido para ingles; lexicos PT sao limitados')
print('3. Negacao implicita: "nao gostei" pode nao ser capturada corretamente')
print('4. Contexto dominio-especifico: girias tecnicas nao no lexicon')

## 2. Analise de Sentimentos com Machine Learning

Abordagem baseada em ML: TF-IDF como representacao de texto + classificadores supervisionados. Comparamos multiplos algoritmos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Dataset sintetico de reviews em ingles (200 exemplos balanceados)
reviews_pos = [
    "This product exceeded all my expectations. Absolutely love it!",
    "Amazing quality, fast shipping and great customer service.",
    "Best purchase I have made in years. Highly recommend!",
    "Perfect fit, great material, very happy with this buy.",
    "Fantastic experience from start to finish. Will buy again.",
    "Outstanding quality and value for money. Five stars!",
    "Incredible product, works exactly as described.",
    "Wonderful experience! The staff were extremely helpful.",
    "Excellent performance and very user friendly interface.",
    "Love this! It solved exactly the problem I had.",
    "Super happy with my purchase. Works perfectly!",
    "Great value, premium quality, fast delivery. Recommend!",
    "One of the best products I own. Very satisfied!",
    "Beautiful design and excellent build quality. Impressed!",
    "Exactly what I needed. Quick delivery and well packaged."
] * 5  # 75 positivos

reviews_neg = [
    "Terrible quality, broke after two days. Complete waste of money.",
    "Do not buy this! Worst product I have ever purchased.",
    "Extremely disappointed. Nothing like the description.",
    "Poor quality and very slow delivery. Very unhappy!",
    "This is garbage. Stopped working after one week.",
    "Awful customer service, they refused to give me a refund.",
    "Horrible experience, item arrived broken and damaged.",
    "Never again! The product is cheap and poorly made.",
    "Complete scam. Item never arrived despite weeks of waiting.",
    "Useless product. The description was totally misleading.",
    "Broken on arrival. Very disappointed with the quality.",
    "Terrible experience from start to finish. Avoid at all costs.",
    "Very poor performance and terrible battery life.",
    "Would not recommend to anyone. Completely useless.",
    "Worst purchase ever. Money totally wasted on this junk."
] * 5  # 75 negativos

reviews_neu = [
    "The product arrived on time and was packaged adequately.",
    "Item is as described. Standard quality for the price.",
    "Delivery was average, product seems ok so far.",
    "Nothing special but it does what it says.",
    "Received the product. Will update after testing it more.",
    "Fairly standard product. Works as expected.",
    "Average product at average price. No complaints, no praise.",
    "It works. Not better or worse than similar products.",
    "Acceptable quality. About what I expected at this price.",
    "Mediocre product but serves its basic purpose."
] * 5  # 50 neutros

texts = reviews_pos + reviews_neg + reviews_neu
labels = (['Positivo'] * len(reviews_pos) +
          ['Negativo'] * len(reviews_neg) +
          ['Neutro'] * len(reviews_neu))

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Treinar Logistic Regression com TF-IDF
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=1)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

print('=== Regressao Logistica + TF-IDF ===')
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Comparar multiplos modelos
modelos = {
    'TF-IDF + Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
    ]),
    'TF-IDF + LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
        ('clf', LinearSVC(C=1.0, max_iter=2000, random_state=42))
    ]),
    'TF-IDF + Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, min_df=1)),
        ('clf', MultinomialNB(alpha=0.5))
    ]),
    'TF-IDF + Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
}

resultados_ml = {}
for nome, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados_ml[nome] = {
        'Acuracia': accuracy_score(y_test, y_pred),
        'F1-macro': f1_score(y_test, y_pred, average='macro')
    }

df_comp = pd.DataFrame(resultados_ml).T.reset_index()
df_comp.columns = ['Modelo', 'Acuracia', 'F1-macro']
print('=== Comparacao de Modelos de Analise de Sentimentos ===')
print(df_comp.to_string(index=False))

# Grafico de comparacao
x = np.arange(len(df_comp))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, df_comp['Acuracia'], width, label='Acuracia',
               color=MINEBLUE, alpha=0.85)
bars2 = ax.bar(x + width/2, df_comp['F1-macro'], width, label='F1-macro',
               color=MINEORANGE, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(' + ', '\n+\n') for m in df_comp['Modelo']], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Comparacao de Modelos - Analise de Sentimentos')
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('comparacao_sentimentos.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. NLP Pipeline - Pre-processamento de Texto

Pipeline completo de NLP: tokenizacao, remocao de stopwords, stemming, lematizacao, n-gramas, TF-IDF e WordCloud.

In [ ]:
import re
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Corpus de exemplo em portugues
corpus_pt = [
    "Mineracao de dados e a area da computacao que extrai conhecimento de grandes conjuntos de dados.",
    "O pre-processamento e fundamental para garantir a qualidade dos dados antes da analise.",
    "Algoritmos de classificacao como arvore de decisao e redes neurais sao amplamente utilizados.",
    "A regressao logistica e um metodo estatistico poderoso para classificacao binaria.",
    "Redes neurais profundas aprendem representacoes hierarquicas de dados complexos.",
    "O agrupamento k-means divide os dados em grupos de acordo com sua similaridade.",
    "Analise de sentimentos permite identificar opinioes positivas e negativas em textos.",
    "Sistemas de recomendacao utilizam filtragem colaborativa para sugerir produtos relevantes.",
    "A visualizacao de dados e essencial para comunicar resultados de mineracao de dados.",
    "Processamento de linguagem natural transforma texto bruto em representacoes numericas.",
    "KDD e o processo de descoberta de conhecimento em bases de dados estruturadas.",
    "Random Forest combina multiplas arvores de decisao para melhorar a robustez do modelo.",
    "Dados desbalanceados exigem tecnicas especificas como SMOTE e class weighting.",
    "A validacao cruzada estima o desempenho do modelo de forma mais confiavel.",
    "Big data envolve processamento de volumes massivos de dados com alta velocidade."
]

stop_pt = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()

def preprocessar_texto(texto):
    """Pipeline completo de pre-processamento para portugues."""
    # 1. Lowercase e remocao de caracteres especiais
    texto = texto.lower()
    texto = re.sub(r'[^a-zacde-mostu-z0-9\s]', '', texto)
    # 2. Tokenizacao
    tokens = word_tokenize(texto, language='portuguese')
    # 3. Remocao de stopwords e tokens curtos
    tokens = [t for t in tokens if t not in stop_pt and len(t) > 2]
    # 4. Stemming com RSLP (algoritmo para portugues)
    stems = [stemmer.stem(t) for t in tokens]
    return tokens, stems

# Aplicar pipeline
corpus_tokens = []
corpus_stems = []
for doc in corpus_pt:
    toks, stms = preprocessar_texto(doc)
    corpus_tokens.extend(toks)
    corpus_stems.extend(stms)

# Frequencia de tokens
freq_tokens = Counter(corpus_tokens).most_common(20)
freq_stems = Counter(corpus_stems).most_common(20)

# Visualizacoes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top 15 tokens
palavras_tok, contagens_tok = zip(*freq_tokens[:15])
axes[0, 0].barh(list(palavras_tok), list(contagens_tok), color=MINEBLUE, alpha=0.8)
axes[0, 0].set_title('Top 15 Tokens (apos stopwords)')
axes[0, 0].invert_yaxis()

# Top 15 stems
palavras_st, contagens_st = zip(*freq_stems[:15])
axes[0, 1].barh(list(palavras_st), list(contagens_st), color=MINEORANGE, alpha=0.8)
axes[0, 1].set_title('Top 15 Stems (RSLP)')
axes[0, 1].invert_yaxis()

# WordCloud dos tokens
freq_dict = dict(freq_tokens)
wc = WordCloud(width=600, height=300, background_color='white',
               colormap='Blues', max_words=50).generate_from_frequencies(freq_dict)
axes[1, 0].imshow(wc, interpolation='bilinear')
axes[1, 0].axis('off')
axes[1, 0].set_title('WordCloud - Corpus NLP')

# TF-IDF heatmap (top 10 termos x 5 documentos)
tfidf = TfidfVectorizer(max_features=10, stop_words=list(stop_pt))
tfidf_matrix = tfidf.fit_transform(corpus_pt[:5]).toarray()
df_tfidf = pd.DataFrame(tfidf_matrix, columns=tfidf.get_feature_names_out(),
                        index=[f'Doc {i+1}' for i in range(5)])
sns.heatmap(df_tfidf, annot=True, fmt='.2f', cmap='Blues',
            ax=axes[1, 1], cbar=True)
axes[1, 1].set_title('TF-IDF - Top 10 Termos x 5 Documentos')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('Pipeline NLP - Pre-processamento de Texto em Portugues', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('nlp_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar exemplo de transformacao
texto_ex = corpus_pt[0]
toks, stms = preprocessar_texto(texto_ex)
print('\n=== EXEMPLO DE PIPELINE NLP ===')
print(f'Original:     {texto_ex}')
print(f'Tokens:       {toks}')
print(f'Stems (RSLP): {stms}')

## 4. Sistema de Recomendacao - Filtragem Colaborativa

Utilizamos a biblioteca Surprise com o dataset MovieLens-100k. Comparamos SVD, SVDpp, KNN e NMF com validacao cruzada.

In [ ]:
from surprise import Dataset, Reader, SVD, SVDpp, NMF
from surprise import KNNBasic, KNNWithMeans
from surprise.model_selection import cross_validate, train_test_split as surp_split
from surprise import accuracy

# Carregar MovieLens-100k
print('Carregando MovieLens-100k...')
data = Dataset.load_builtin('ml-100k')

# Comparar algoritmos com cross-validation (3 folds)
algoritmos = {
    'SVD': SVD(n_factors=50, n_epochs=20, random_state=42),
    'SVD++': SVDpp(n_factors=20, n_epochs=10, random_state=42),
    'NMF': NMF(n_factors=15, n_epochs=50, random_state=42),
    'KNN Basic': KNNBasic(k=20, sim_options={'name': 'cosine', 'user_based': True}),
    'KNN Means': KNNWithMeans(k=20, sim_options={'name': 'pearson', 'user_based': True}),
}

resultados_surp = {}
print('Treinando e validando algoritmos...')
for nome, algo in algoritmos.items():
    cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=3,
                                 verbose=False, n_jobs=-1)
    resultados_surp[nome] = {
        'RMSE': cv_results['test_rmse'].mean(),
        'MAE': cv_results['test_mae'].mean(),
        'RMSE_std': cv_results['test_rmse'].std()
    }
    print(f'  {nome:12s} -> RMSE: {resultados_surp[nome]["RMSE"]:.4f} '
          f'(+/-{resultados_surp[nome]["RMSE_std"]:.4f}) | '
          f'MAE: {resultados_surp[nome]["MAE"]:.4f}')

# Visualizar comparacao
df_surp = pd.DataFrame(resultados_surp).T.reset_index()
df_surp.columns = ['Algoritmo', 'RMSE', 'MAE', 'RMSE_std']
df_surp = df_surp.sort_values('RMSE')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(df_surp['Algoritmo'], df_surp['RMSE'],
             xerr=df_surp['RMSE_std'], color=MINEBLUE, alpha=0.8, capsize=5)
axes[0].set_title('RMSE por Algoritmo (menor e melhor)')
axes[0].set_xlabel('RMSE (3-fold CV)')

axes[1].barh(df_surp['Algoritmo'], df_surp['MAE'], color=MINEORANGE, alpha=0.8)
axes[1].set_title('MAE por Algoritmo (menor e melhor)')
axes[1].set_xlabel('MAE (3-fold CV)')

plt.suptitle('Comparacao de Algoritmos - Sistema de Recomendacao (ML-100k)', fontsize=13)
plt.tight_layout()
plt.savefig('comparacao_recomendacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from surprise import Dataset

# Treinar SVD no conjunto completo e gerar recomendacoes
trainset = data.build_full_trainset()
svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)

# Gerar Top-10 recomendacoes para usuario ID=196
user_id = '196'
testset = trainset.build_anti_testset()  # pares nao vistos pelo usuario
testset_user = [(uid, iid, r) for uid, iid, r in testset if uid == user_id]

predicoes = svd.test(testset_user)
predicoes_sorted = sorted(predicoes, key=lambda x: x.est, reverse=True)[:10]

print(f'=== TOP-10 RECOMENDACOES PARA USUARIO {user_id} (SVD) ===')
print(f'{"Pos":>4} {"Item ID":>8} {"Rating Previsto":>15}')
print('-' * 35)
for i, pred in enumerate(predicoes_sorted, 1):
    print(f'{i:>4} {pred.iid:>8} {pred.est:>15.3f}')

# Estatisticas dos ratings previstos
ratings_previstos = [p.est for p in predicoes_sorted]
print(f'\nRating previsto medio (Top-10): {np.mean(ratings_previstos):.3f}')
print(f'Range: [{min(ratings_previstos):.3f}, {max(ratings_previstos):.3f}]')

# Histograma dos ratings previstos
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 11), ratings_previstos, color=MINEGREEN, alpha=0.8)
ax.set_xlabel('Posicao no Ranking')
ax.set_ylabel('Rating Previsto (1-5)')
ax.set_title(f'Top-10 Recomendacoes SVD - Usuario {user_id}')
ax.set_xticks(range(1, 11))
ax.set_ylim(0, 5.5)
for i, r in enumerate(ratings_previstos, 1):
    ax.text(i, r + 0.1, f'{r:.2f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Sistema de Recomendacao - Filtragem por Conteudo

Recomendacao baseada em atributos dos proprios itens (descricao, generos, tags) usando TF-IDF e similaridade por cosseno.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Dataset de filmes com metadados
filmes = pd.DataFrame({
    'titulo': [
        'Matrix', 'Matrix Reloaded', 'Matrix Revolutions', 'Inception',
        'Interstellar', 'Blade Runner 2049', 'Ex Machina', 'Ghost in the Shell',
        'The Terminator', 'Terminator 2', 'RoboCop', 'Total Recall',
        'The Dark Knight', 'Batman Begins', 'Iron Man', 'Avengers',
        'Pulp Fiction', 'Kill Bill', 'Django Unchained', 'Inglourious Basterds'
    ],
    'descricao': [
        'hacker discovers reality is a simulation controlled by machines artificial intelligence cyberpunk sci-fi action',
        'neo returns to matrix machines war simulation artificial intelligence action cyberpunk sequel',
        'final battle machines humans war matrix artificial intelligence simulation sci-fi action',
        'thief enters dreams subconscious heist sci-fi thriller psychological reality layers',
        'astronauts travel wormhole space time black hole quantum physics survival family love sci-fi',
        'replicant blade runner mystery noir cyberpunk future android artificial intelligence sci-fi',
        'programmer isolated facility artificial intelligence robot consciousness experiment thriller sci-fi',
        'cyborg policewoman hacker artificial intelligence cyberpunk future android identity thriller',
        'cyborg assassin time travel future war machines humans survival action sci-fi thriller',
        'cyborg terminator future war humans machines survival action sequel sci-fi',
        'police robot corporation future crime action sci-fi cyberpunk thriller',
        'man memory implant mars colony future sci-fi action thriller conspiracy',
        'batman joker villain crime hero thriller action hero city corruption',
        'bruce wayne batman origin crime hero thriller action training detective',
        'billionaire iron suit hero villain action superhero technology weapons',
        'superheroes team villain battle earth save action sci-fi superhero ensemble',
        'hitmen gangsters crime Los Angeles dance drama thriller nonlinear narrative',
        'revenge assassin martial arts action drama thriller female protagonist',
        'slave revenge outlaw western action drama thriller freedom justice',
        'war nazi revenge spy thriller drama historical suspense'
    ]
})

# TF-IDF sobre as descricoes
tfidf_cb = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix_cb = tfidf_cb.fit_transform(filmes['descricao'])

# Matriz de similaridade cosseno
sim_matrix = cosine_similarity(tfidf_matrix_cb)

def recomendar_por_conteudo(titulo, n=5):
    """Retorna os N filmes mais similares ao titulo dado."""
    if titulo not in filmes['titulo'].values:
        return f'Filme "{titulo}" nao encontrado.'
    idx = filmes[filmes['titulo'] == titulo].index[0]
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [(i, s) for i, s in scores if i != idx][:n]
    return [(filmes.iloc[i]['titulo'], round(s, 3)) for i, s in scores]

# Testar recomendacoes
for filme_teste in ['Matrix', 'The Dark Knight', 'Pulp Fiction']:
    recs = recomendar_por_conteudo(filme_teste, n=5)
    print(f'\nFilmes similares a "{filme_teste}":')
    for titulo, sim in recs:
        print(f'  {titulo:30s} (similaridade: {sim:.3f})')

# Visualizar heatmap de similaridade
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(sim_matrix, dtype=bool)
np.fill_diagonal(mask, True)
sns.heatmap(sim_matrix, xticklabels=filmes['titulo'],
            yticklabels=filmes['titulo'], cmap='YlOrRd',
            mask=mask, ax=ax, vmin=0, vmax=0.8)
ax.set_title('Matriz de Similaridade - Filtragem por Conteudo (TF-IDF + Cosseno)')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('similaridade_conteudo.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Social CRM - Segmentacao de Clientes (RFM)

Criamos um dataset sintetico de e-commerce, calculamos RFM, aplicamos K-Means e criamos perfis de segmento.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Gerar dataset sintetico de e-commerce (1000 clientes)
np.random.seed(42)
n_clientes = 1000
hoje = datetime(2024, 6, 15)

# Perfis de clientes com distribuicoes diferentes
clientes_dados = []
for i in range(n_clientes):
    tipo = np.random.choice(['campiao','leal','em_risco','perdido'], p=[0.15,0.30,0.30,0.25])
    if tipo == 'campiao':
        recency = int(np.random.exponential(7) + 1)
        frequency = int(np.random.normal(25, 5))
        monetary = np.random.normal(800, 200)
    elif tipo == 'leal':
        recency = int(np.random.exponential(20) + 5)
        frequency = int(np.random.normal(15, 4))
        monetary = np.random.normal(400, 100)
    elif tipo == 'em_risco':
        recency = int(np.random.normal(90, 30))
        frequency = int(np.random.normal(8, 3))
        monetary = np.random.normal(250, 80)
    else:  # perdido
        recency = int(np.random.normal(200, 50))
        frequency = int(np.random.normal(3, 2))
        monetary = np.random.normal(100, 50)

    clientes_dados.append({
        'cliente_id': f'CLI{i:04d}',
        'Recency': max(1, recency),
        'Frequency': max(1, frequency),
        'Monetary': max(10.0, monetary)
    })

rfm_df = pd.DataFrame(clientes_dados)

print('=== ESTATISTICAS RFM ===')
print(rfm_df[['Recency','Frequency','Monetary']].describe().round(2))

# Normalizar RFM
scaler_rfm = StandardScaler()
rfm_scaled = scaler_rfm.fit_transform(rfm_df[['Recency','Frequency','Monetary']])

# Metodo do cotovelo
inertias = []
K_range = range(2, 11)
for k in K_range:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(rfm_scaled)
    inertias.append(km_temp.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-', markersize=8)
axes[0].axvline(x=4, color=MINERED, linestyle='--', alpha=0.7, label='k=4 selecionado')
axes[0].set_xlabel('Numero de Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Metodo do Cotovelo - Selecao de k')
axes[0].legend()

# Treinar K-Means com k=4
km_rfm = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_df['Segmento'] = km_rfm.fit_predict(rfm_scaled)

# Perfil dos segmentos
perfil = rfm_df.groupby('Segmento')[['Recency','Frequency','Monetary']].mean().round(1)
print('\n=== PERFIL DOS SEGMENTOS ===')
print(perfil)

# Tamanho dos segmentos
tamanho = rfm_df['Segmento'].value_counts().sort_index()
axes[1].pie(tamanho, labels=[f'Seg {i}' for i in tamanho.index],
            autopct='%1.1f%%', colors=[MINEBLUE, MINEGREEN, MINEORANGE, MINERED],
            startangle=90)
axes[1].set_title('Distribuicao de Clientes por Segmento')
plt.tight_layout()
plt.savefig('rfm_segmentacao.png', dpi=150, bbox_inches='tight')
plt.show()

# Nomear segmentos baseado nos centroides
centroides = pd.DataFrame(scaler_rfm.inverse_transform(km_rfm.cluster_centers_),
                          columns=['Recency','Frequency','Monetary'])
centroides['Recency_rank'] = centroides['Recency'].rank()
centroides['Frequency_rank'] = centroides['Frequency'].rank(ascending=False)
nomes = {}
for i in centroides.index:
    r, f = centroides.loc[i, 'Recency_rank'], centroides.loc[i, 'Frequency_rank']
    if r <= 2 and f <= 2:
        nomes[i] = 'Campiao'
    elif r <= 2:
        nomes[i] = 'Novato Promissor'
    elif f <= 2:
        nomes[i] = 'Em Risco'
    else:
        nomes[i] = 'Inativo'
rfm_df['Nome_Segmento'] = rfm_df['Segmento'].map(nomes)

# Estrategias de marketing
estrategias = {
    'Campiao': 'Recompensar com programa VIP, solicitar reviews, early access a lancamentos',
    'Novato Promissor': 'Incentivar segunda compra, enviar cupom de desconto progressivo',
    'Em Risco': 'Campanha de reativacao: email personalizado com oferta especial urgente',
    'Inativo': 'Tentativa de win-back com grande desconto; se sem resposta, remover da lista'
}
print('\n=== ESTRATEGIAS DE MARKETING POR SEGMENTO ===')
for seg, estrategia in estrategias.items():
    print(f'  {seg:20s}: {estrategia}')

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

# Previsao de Churn com base em RFM + features de engajamento
np.random.seed(42)

# Adicionar features de engajamento ao dataset
rfm_df['dias_desde_registro'] = np.random.randint(100, 1500, n_clientes)
rfm_df['tickets_suporte'] = np.random.poisson(1.5, n_clientes)
rfm_df['avg_order_value'] = rfm_df['Monetary'] / rfm_df['Frequency']
rfm_df['recency_norm'] = rfm_df['Recency'] / rfm_df['dias_desde_registro']

# Definir churn: clientes com Recency > 120 dias E Frequency < 5
rfm_df['churn'] = ((rfm_df['Recency'] > 120) & (rfm_df['Frequency'] < 5)).astype(int)
print(f'Taxa de churn: {rfm_df["churn"].mean():.1%}')

features_churn = ['Recency', 'Frequency', 'Monetary', 'tickets_suporte',
                  'avg_order_value', 'recency_norm']
X_churn = rfm_df[features_churn]
y_churn = rfm_df['churn']

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_churn, y_churn, test_size=0.2,
                                            random_state=42, stratify=y_churn)

# XGBoost (usando GradientBoosting do sklearn)
try:
    from xgboost import XGBClassifier
    clf_churn = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4,
                               use_label_encoder=False, eval_metric='logloss',
                               random_state=42)
except ImportError:
    clf_churn = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                            max_depth=4, random_state=42)

clf_churn.fit(X_tr, y_tr)
y_proba = clf_churn.predict_proba(X_te)[:, 1]
y_pred_ch = clf_churn.predict(X_te)

auc = roc_auc_score(y_te, y_proba)
fpr, tpr, _ = roc_curve(y_te, y_proba)

print(f'\nAUC-ROC: {auc:.4f}')
print(classification_report(y_te, y_pred_ch))

# Visualizacoes de churn
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
MINEGRAY = '#7f7f7f'
# ROC Curve
axes[0].plot(fpr, tpr, color=MINEBLUE, lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1], color=MINEGRAY if hasattr(plt.cm,'gray') else 'gray',
             linestyle='--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC - Previsao de Churn')
axes[0].legend()

# Feature Importance
importances = clf_churn.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
axes[1].barh(range(len(features_churn)),
             importances[sorted_idx][::-1],
             color=MINEORANGE, alpha=0.8)
axes[1].set_yticks(range(len(features_churn)))
axes[1].set_yticklabels([features_churn[i] for i in sorted_idx][::-1])
axes[1].set_title('Importancia das Features - Churn')
axes[1].set_xlabel('Importancia')

# Score de risco dos clientes (Top 20% em risco)
rfm_df_test = rfm_df.iloc[y_te.index]
rfm_df_test = rfm_df_test.copy()
rfm_df_test['churn_prob'] = y_proba
top20_risco = rfm_df_test.nlargest(int(len(rfm_df_test) * 0.2), 'churn_prob')
axes[2].hist(y_proba, bins=20, color=MINEBLUE, alpha=0.7, edgecolor='white')
axes[2].axvline(x=np.percentile(y_proba, 80), color=MINERED, linestyle='--',
                label='Limiar Top-20% risco')
axes[2].set_xlabel('Probabilidade de Churn')
axes[2].set_ylabel('Quantidade de Clientes')
axes[2].set_title('Distribuicao de Probabilidade de Churn')
axes[2].legend()

plt.suptitle('Previsao de Churn com Gradient Boosting', fontsize=13)
plt.tight_layout()
plt.savefig('churn_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop-20% clientes em risco: {len(top20_risco)} clientes')
print(f'Acao recomendada: campanha de retencao personalizada para esses {len(top20_risco)} clientes')

## 7. Web Mining - Analise de Texto e PageRank

Implementamos PageRank por iteracao de potencia e comparamos com NetworkX.

In [ ]:
import networkx as nx

# Grafo web simulado: 10 paginas com hiperlinks
# Topologia: pagina 0 e hub central, paginas 1-4 sao conteudo, 5-9 sao folhas
edges = [
    (0, 1), (0, 2), (0, 3), (0, 4),  # hub para conteudo
    (1, 2), (1, 5), (1, 6),           # conteudo interligado
    (2, 3), (2, 7),
    (3, 4), (3, 8),
    (4, 0), (4, 9),                    # folha apontando de volta
    (5, 0), (6, 0), (7, 0), (8, 1), (9, 2),  # folhas apontando para hub
    (5, 7), (6, 8), (7, 9)             # links entre folhas
]

N = 10  # numero de paginas
paginas = [f'Pagina_{i}' for i in range(N)]

# Implementacao do PageRank por iteracao de potencia
def pagerank_manual(edges, N, d=0.85, max_iter=100, tol=1e-6):
    """Implementacao do PageRank por iteracao de potencia."""
    # Inicializar
    PR = np.ones(N) / N
    
    # Construir dicionario de out-links
    out_links = {i: [] for i in range(N)}
    in_links = {i: [] for i in range(N)}
    for src, dst in edges:
        out_links[src].append(dst)
        in_links[dst].append(src)
    
    historico = [PR.copy()]
    for iteration in range(max_iter):
        PR_new = np.zeros(N)
        for p in range(N):
            soma = 0
            for q in in_links[p]:
                if len(out_links[q]) > 0:
                    soma += PR[q] / len(out_links[q])
                else:
                    soma += PR[q] / N  # dangling node
            PR_new[p] = (1 - d) / N + d * soma
        
        diff = np.abs(PR_new - PR).sum()
        PR = PR_new
        historico.append(PR.copy())
        if diff < tol:
            print(f'Convergiu em {iteration + 1} iteracoes (diff={diff:.2e})')
            break
    return PR, historico

# Calcular PageRank manual
pr_manual, historico = pagerank_manual(edges, N, d=0.85)

# Calcular com NetworkX
G = nx.DiGraph()
G.add_nodes_from(range(N))
G.add_edges_from(edges)
pr_nx = nx.pagerank(G, alpha=0.85)
pr_nx_array = np.array([pr_nx[i] for i in range(N)])

# Comparar resultados
print('\n=== COMPARACAO PAGERANK: MANUAL vs NETWORKX ===')
print(f'{"Pagina":>12} {"PR Manual":>12} {"PR NetworkX":>12} {"Diferenca":>12}')
print('-' * 50)
for i in range(N):
    diff = abs(pr_manual[i] - pr_nx_array[i])
    print(f'{paginas[i]:>12} {pr_manual[i]:>12.6f} {pr_nx_array[i]:>12.6f} {diff:>12.2e}')
print(f'\nErro maximo: {np.max(np.abs(pr_manual - pr_nx_array)):.2e}')

# Visualizar grafo com tamanho proporcional ao PageRank
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pos = nx.spring_layout(G, seed=42, k=2)
node_sizes = [pr_nx[i] * 8000 for i in range(N)]
node_colors = [pr_nx[i] for i in range(N)]

nx.draw_networkx(G, pos=pos, ax=axes[0],
                 node_size=node_sizes, node_color=node_colors,
                 cmap=plt.cm.Blues, edge_color='gray', alpha=0.9,
                 labels={i: f'P{i}\n{pr_nx[i]:.3f}' for i in range(N)},
                 font_size=7, font_color='black',
                 arrows=True, arrowsize=15)
axes[0].set_title('Grafo Web - Tamanho proporcional ao PageRank')
axes[0].axis('off')

# Convergencia do PageRank
historico_array = np.array(historico)
for i in range(N):
    axes[1].plot(historico_array[:, i], alpha=0.7, label=f'P{i}' if i < 5 else None)
axes[1].set_xlabel('Iteracao')
axes[1].set_ylabel('PageRank Score')
axes[1].set_title('Convergencia do PageRank por Iteracao')
axes[1].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig('pagerank_grafo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from bs4 import BeautifulSoup

# Simulacao de web scraping com HTML estatico (sem requisicao real)
html_simulado = """
<html>
<head><title>Noticias de Tecnologia - Exemplo</title></head>
<body>
  <h1>Portal de Noticias Tecnologicas</h1>
  <article class='noticia'>
    <h2>Inteligencia Artificial supera medicos em diagnostico de cancer</h2>
    <p>Pesquisadores desenvolveram um algoritmo de deep learning que detecta
    cancer de pele com acuracia superior a dermatologistas experientes.
    O modelo foi treinado com mais de 100.000 imagens clinicas.</p>
    <span class='data'>15/06/2024</span>
  </article>
  <article class='noticia'>
    <h2>Nova versao do Python traz melhorias de performance significativas</h2>
    <p>A linguagem de programacao Python lancou sua nova versao com otimizacoes
    que aumentam a velocidade de execucao em ate 60% em benchmarks padrao.
    A comunidade open source celebra o lancamento.</p>
    <span class='data'>14/06/2024</span>
  </article>
  <article class='noticia'>
    <h2>Empresa brasileira de fintech capta R$ 500 milhoes em rodada Serie C</h2>
    <p>A startup de tecnologia financeira anunciou captacao record no Brasil.
    O investimento sera usado para expansao internacional e desenvolvimento
    de novos produtos de credito baseados em machine learning.</p>
    <span class='data'>13/06/2024</span>
  </article>
</body>
</html>
"""

soup = BeautifulSoup(html_simulado, 'html.parser')

# Extrair dados
noticias = []
for article in soup.find_all('article', class_='noticia'):
    titulo = article.find('h2').get_text(strip=True)
    conteudo = article.find('p').get_text(strip=True)
    data = article.find('span', class_='data').get_text(strip=True)
    noticias.append({'titulo': titulo, 'conteudo': conteudo, 'data': data})

df_noticias = pd.DataFrame(noticias)
print('=== NOTICIAS EXTRAIDAS (Web Scraping Simulado) ===')
for _, row in df_noticias.iterrows():
    print(f'\n[{row["data"]}] {row["titulo"]}')
    print(f'  {row["conteudo"][:100]}...')

# Analise de sentimentos VADER sobre os titulos
print('\n=== ANALISE DE SENTIMENTOS DOS TITULOS ===')
for _, row in df_noticias.iterrows():
    scores = analyzer.polarity_scores(row['titulo'])
    compound = scores['compound']
    sentimento = 'Positivo' if compound >= 0.05 else ('Negativo' if compound <= -0.05 else 'Neutro')
    print(f'  Compound={compound:+.3f} [{sentimento}]: {row["titulo"][:60]}')

## 8. Mineracao de Redes Sociais

Geramos uma rede sintetica com modelo Barabasi-Albert, calculamos medidas de centralidade e detectamos comunidades.

In [ ]:
import networkx as nx
from collections import defaultdict

# Gerar rede social com modelo Barabasi-Albert (preferential attachment)
np.random.seed(42)
G_social = nx.barabasi_albert_graph(n=150, m=2, seed=42)

# Calcular medidas de centralidade
degree_cent = nx.degree_centrality(G_social)
betweenness_cent = nx.betweenness_centrality(G_social, normalized=True)
closeness_cent = nx.closeness_centrality(G_social)
eigenvector_cent = nx.eigenvector_centrality(G_social, max_iter=500)

# DataFrame de centralidades
df_cent = pd.DataFrame({
    'no': list(G_social.nodes()),
    'degree': [degree_cent[n] for n in G_social.nodes()],
    'betweenness': [betweenness_cent[n] for n in G_social.nodes()],
    'closeness': [closeness_cent[n] for n in G_social.nodes()],
    'eigenvector': [eigenvector_cent[n] for n in G_social.nodes()]
})

print('=== TOP-10 NOS POR GRAU DE CENTRALIDADE ===')
top10 = df_cent.sort_values('degree', ascending=False).head(10)
print(top10.to_string(index=False))

# Deteccao de comunidades (Greedy Modularity)
from networkx.algorithms.community import greedy_modularity_communities
communities = list(greedy_modularity_communities(G_social))
print(f'\nComunidades detectadas: {len(communities)}')
for i, comm in enumerate(communities[:5]):
    print(f'  Comunidade {i+1}: {len(comm)} nos')

# Mapa de comunidade por no
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i

# Visualizacoes
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

# Grafo colorido por comunidade
pos_social = nx.spring_layout(G_social, seed=42, k=0.3)
colors_comm = [community_map[n] for n in G_social.nodes()]
sizes_cent = [degree_cent[n] * 800 + 30 for n in G_social.nodes()]
nx.draw_networkx(G_social, pos=pos_social, ax=axes[0],
                 node_color=colors_comm, cmap=plt.cm.tab10,
                 node_size=sizes_cent, with_labels=False,
                 edge_color='lightgray', alpha=0.8, arrows=False)
axes[0].set_title(f'Rede Social - {len(communities)} Comunidades')
axes[0].axis('off')

# Distribuicao de grau (lei de potencia)
degrees = [d for _, d in G_social.degree()]
degree_count = np.bincount(degrees)
k_vals = np.where(degree_count > 0)[0]
p_vals = degree_count[k_vals]
axes[1].loglog(k_vals, p_vals, 'bo', markersize=6, alpha=0.7)
axes[1].set_xlabel('Grau k (log)')
axes[1].set_ylabel('Frequencia P(k) (log)')
axes[1].set_title('Distribuicao de Grau (Lei de Potencia - Barabasi-Albert)')
axes[1].grid(True, which='both', alpha=0.3)

# Correlacao entre medidas de centralidade
sns.heatmap(df_cent[['degree','betweenness','closeness','eigenvector']].corr(),
            annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[2],
            vmin=-1, vmax=1, center=0)
axes[2].set_title('Correlacao entre Medidas de Centralidade')

plt.suptitle('Analise de Rede Social - Modelo Barabasi-Albert', fontsize=13)
plt.tight_layout()
plt.savefig('rede_social.png', dpi=150, bbox_inches='tight')
plt.show()

# Estatisticas da rede
print('\n=== METRICAS GLOBAIS DA REDE ===')
print(f'Nos:                    {G_social.number_of_nodes()}')
print(f'Arestas:                {G_social.number_of_edges()}')
print(f'Densidade:              {nx.density(G_social):.4f}')
print(f'Coef. de agrupamento:   {nx.average_clustering(G_social):.4f}')
print(f'Diametro (aprox):       {nx.diameter(G_social)}')
print(f'Caminho medio:          {nx.average_shortest_path_length(G_social):.3f}')
print(f'Modularidade (Louvain): estimada com {len(communities)} comunidades')

## 9. Exercicios

Resolva os exercicios abaixo para consolidar o aprendizado do Modulo 7.

In [ ]:
# =============================================================
# EXERCICIO 1: Classificador de Sentimentos com BERT
# =============================================================
# Objetivo: Usar um modelo BERT pre-treinado para analise de sentimentos
#
# Instrucoes:
# 1. Carregue o modelo 'nlptown/bert-base-multilingual-uncased-sentiment'
#    do Hugging Face (suporta portugues, 5 estrelas)
# 2. Use o pipeline de transformers para classificar as 20 frases
#    do Exercicio 1 da secao VADER
# 3. Compare os resultados com VADER em uma tabela
# 4. Calcule e compare as acuracias

# Seu codigo aqui:
# from transformers import pipeline
# sentiment_bert = pipeline('sentiment-analysis',
#                            model='nlptown/bert-base-multilingual-uncased-sentiment')
# ...

print('EXERCICIO 1: Implemente o classificador BERT para sentimentos')
print('Dica: Use transformers.pipeline(task="sentiment-analysis", model=...)')
print('Documente as diferencas entre VADER e BERT para frases em portugues')


# =============================================================
# EXERCICIO 2: Novo Algoritmo de Recomendacao
# =============================================================
# Objetivo: Adicionar e avaliar um novo algoritmo ao comparativo do Modulo
#
# Instrucoes:
# 1. Implemente o algoritmo 'CoClustering' da biblioteca Surprise
# 2. Adicione-o ao dicionario 'algoritmos' da Secao 4
# 3. Compare RMSE e MAE com os demais algoritmos
# 4. Discuta: em quais cenarios CoClustering pode superar SVD?

print('\nEXERCICIO 2: Adicione CoClustering ao comparativo Surprise')
print('from surprise import CoClustering')
print('Experimente variar n_cltr_u e n_cltr_i para otimizar')


# =============================================================
# EXERCICIO 3: Analise de Dados de Redes Sociais Proprios
# =============================================================
# Objetivo: Aplicar as tecnicas do modulo a dados reais de redes sociais
#
# Instrucoes:
# 1. Exporte seus dados de seguidores/seguidos de uma rede social
#    (Twitter API, Instagram Basic Display API, ou LinkedIn)
# 2. Construa um grafo de conexoes com NetworkX
# 3. Calcule as 4 medidas de centralidade
# 4. Detecte comunidades e interprete o que elas representam
# 5. Identifique os 5 nos mais influentes e justifique com as metricas
#
# Alternativa sem dados reais:
# Use o grafo de citacoes academicas disponivel em:
# nx.generators.social.karate_club_graph() (clube de karate de Zachary)

print('\nEXERCICIO 3: Analise de Rede Social')
print('Dataset sugerido: Karate Club (Zachary, 1977)')
G_karate = nx.karate_club_graph()
print(f'Grafo Karate: {G_karate.number_of_nodes()} nos, {G_karate.number_of_edges()} arestas')
print('Aplique: centralidade, deteccao de comunidades, visualizacao')